In [1]:
from unittest import result

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# pip install openai pinecone pandas
from groq import Groq
import cohere
import os
import pandas as pd      # turns text into vectors (the embedding model)
from pinecone import Pinecone, ServerlessSpec   # stores vectors and searches them

oa = cohere.Client(os.environ["COHERE_API_KEY"])
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

EMBED_MODEL = "embed-english-v3.0"
  # 1536 dims — must be the SAME model for every vector in this index
INDEX_NAME = "vidyasankalp-ecom-products"

def embed_batch(texts):
    response = oa.embed(
        texts=texts,
        model=EMBED_MODEL,
        input_type="search_document"
    )
    return response.embeddings


In [3]:
DATA_URL="products.csv"
df = pd.read_csv(DATA_URL, sep="|")
df

,Id,Name,Description,Price,PriceCurrency,SupplyAbility,MinimumOrder
0,d2559c95-bd28-49d8-b53a-538c34a25bcb,Saucony Men's Kinvara 13 Running Shoe,"When it comes to lightweight speed, nothing cr...",600.93,USD,396,574
1,c0717df8-fd27-4355-9c06-a8ad1ebb3e95,Accutire MS-4021B Digital Tire Pressure Gauge ...,About this item Heavy duty construction and ru...,476.98,USD,288,993
2,29ea5156-8582-4345-8672-65a8f4c1e30c,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...,This extraordinary fusion is designed to nouri...,216.74,EUR,640,707
3,cb17beb7-30f0-4857-8793-4a6df0983fa3,"crysting 13 Inch Sewing Box Three Layers, Plas...",About this item 🌟 High Quality Sewing Box 🌟 Ma...,84.40,EUR,85,923
4,b2d75b67-64f3-4ecd-82e6-47ff8e934c37,"Ridgid 62990 T-201 5"" Straight Auger","The product is 62990 AUGER, T201 STRAIGHT 5/8 ...",219.11,EUR,350,120
...,...,...,...,...,...,...,...
746,fc28844f-d741-4ac5-9fc2-d268b17810f4,Whirlpool 28 cu. ft. French Door Refrigerator,The Whirlpool 28 cu. ft. French Door Refrigera...,2199.99,USD,800,2
747,acfdc81f-06be-4c4e-a094-1cc35a24c227,VELCRO Brand - 91302 Thin Clear Dots with Adhe...,VELCRO Brand Thin Clear fasteners blend into g...,356.11,EUR,514,710
748,5ac4fd2c-0681-4de8-aaf4-dd8684436039,Madison Park Amherst Faux Silk Comforter Set-C...,The Madison Park Amherst 7 Piece Comforter Set...,383.20,EUR,831,960
749,d5e8517e-ee02-4d1d-bf1d-f7590e48a0d6,"BIC Gel-Ocity Gel Pens, Medium Point Retractab...",Gel-ocity is a smooth writing retractable gel ...,420.91,EUR,950,503


In [5]:
df['Description']=df['Description'].fillna("")
print(df.shape)
df.head

(751, 7)


<bound method NDFrame.head of                                        Id  \
0    d2559c95-bd28-49d8-b53a-538c34a25bcb   
1    c0717df8-fd27-4355-9c06-a8ad1ebb3e95   
2    29ea5156-8582-4345-8672-65a8f4c1e30c   
3    cb17beb7-30f0-4857-8793-4a6df0983fa3   
4    b2d75b67-64f3-4ecd-82e6-47ff8e934c37   
..                                    ...   
746  fc28844f-d741-4ac5-9fc2-d268b17810f4   
747  acfdc81f-06be-4c4e-a094-1cc35a24c227   
748  5ac4fd2c-0681-4de8-aaf4-dd8684436039   
749  d5e8517e-ee02-4d1d-bf1d-f7590e48a0d6   
750  dbaf168e-25bc-4dae-8fd0-84f00bca9e70   

                                                  Name  \
0                Saucony Men's Kinvara 13 Running Shoe   
1    Accutire MS-4021B Digital Tire Pressure Gauge ...   
2    SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...   
3    crysting 13 Inch Sewing Box Three Layers, Plas...   
4                 Ridgid 62990 T-201 5" Straight Auger   
..                                                 ...   
746      Whirlpool 28 c

In [6]:
df['text']=df['Name']+':'+df['Description']
df.head(3)

,Id,Name,Description,Price,PriceCurrency,SupplyAbility,MinimumOrder,text
0,d2559c95-bd28-49d8-b53a-538c34a25bcb,Saucony Men's Kinvara 13 Running Shoe,"When it comes to lightweight speed, nothing cr...",600.93,USD,396,574,Saucony Men's Kinvara 13 Running Shoe:When it ...
1,c0717df8-fd27-4355-9c06-a8ad1ebb3e95,Accutire MS-4021B Digital Tire Pressure Gauge ...,About this item Heavy duty construction and ru...,476.98,USD,288,993,Accutire MS-4021B Digital Tire Pressure Gauge ...
2,29ea5156-8582-4345-8672-65a8f4c1e30c,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...,This extraordinary fusion is designed to nouri...,216.74,EUR,640,707,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...


In [7]:
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(INDEX_NAME)

In [8]:
BATCH=100
for start in range(0,len(df),BATCH):
    # print(start)
    print(start,start+BATCH)
    chunk=df.iloc[start:start+BATCH]
    print(chunk.head())

0 100
                                     Id  \
0  d2559c95-bd28-49d8-b53a-538c34a25bcb   
1  c0717df8-fd27-4355-9c06-a8ad1ebb3e95   
2  29ea5156-8582-4345-8672-65a8f4c1e30c   
3  cb17beb7-30f0-4857-8793-4a6df0983fa3   
4  b2d75b67-64f3-4ecd-82e6-47ff8e934c37   

                                                Name  \
0              Saucony Men's Kinvara 13 Running Shoe   
1  Accutire MS-4021B Digital Tire Pressure Gauge ...   
2  SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...   
3  crysting 13 Inch Sewing Box Three Layers, Plas...   
4               Ridgid 62990 T-201 5" Straight Auger   

                                         Description   Price PriceCurrency  \
0  When it comes to lightweight speed, nothing cr...  600.93           USD   
1  About this item Heavy duty construction and ru...  476.98           USD   
2  This extraordinary fusion is designed to nouri...  216.74           EUR   
3  About this item 🌟 High Quality Sewing Box 🌟 Ma...   84.40           EUR   
4  The p

In [9]:
def to_metadata(row):
    return {
        "text": row["text"],
        "price": float(row["Price"]),
        "currency": row["PriceCurrency"],
        "supply_ability": int(row["SupplyAbility"]),
        "minimum_order": int(row["MinimumOrder"]),
    }

In [10]:
BATCH = 100
for start in range(0, len(df), BATCH):
    print(start, start + BATCH)
    chunk_df = df.iloc[start:start + BATCH]
    text_list = chunk_df["text"].tolist()
    vectors = embed_batch(text_list)
    ids = chunk_df["Id"].astype(str).tolist()
    metadata_rows = [to_metadata(row) for (row_id,row) in chunk_df.iterrows()]
    batch = [{"id":id, 'values':vector, "metadata":metadata} for id, vector, metadata in zip(ids, vectors, metadata_rows)]
    index.upsert(vectors=batch)
    print("-------------------")

0 100
-------------------
100 200
-------------------
200 300
-------------------
300 400
-------------------
400 500
-------------------
500 600
-------------------
600 700
-------------------
700 800
-------------------


In [12]:
query_vector=embed_batch(['wireless noise-canceling headphone'])[0]
print(query_vector)
result=index.query(vector=query_vector,top_k=5,include_metadata=True)
for match in result['matches']:
    m=match['metadata']
    print(f"{match['score']:.3f}{m['text'][:70]:70s}{m['price']:>8.2f}{m['currency']}")

[-0.019210815, -0.011100769, -0.03491211, 0.022903442, -0.051818848, -0.021118164, 0.0065727234, -0.010673523, -0.03652954, 0.042755127, -0.011131287, 0.006801605, 0.037475586, -0.03189087, 0.06713867, 0.0095825195, 0.020935059, 0.021469116, 0.0019817352, -0.01776123, -0.01701355, 0.002670288, -0.0030117035, -0.013145447, -0.035491943, 0.0385437, -0.04812622, -0.05105591, -0.023590088, -0.04208374, -0.017105103, 0.05807495, 0.017028809, 0.05166626, -0.019424438, 0.053588867, -0.03201294, 0.028411865, 0.010719299, 0.0011358261, 0.02758789, -0.010238647, -0.010177612, 0.028274536, -0.043395996, -0.033172607, 0.020492554, 0.014457703, 0.018112183, 0.013381958, -0.0072898865, -0.014343262, 0.006996155, 0.00969696, -0.026565552, -0.006702423, -0.0259552, -0.009101868, 0.009223938, 0.014160156, -0.0043640137, 0.0368042, 0.022216797, 0.008987427, 0.022369385, -0.03741455, -0.03010559, 0.023422241, -0.003643036, 0.037628174, 0.033325195, -0.028778076, 0.0016641617, 0.02281189, -0.0015583038, -

In [13]:
usd_sample=df[df['PriceCurrency']=="USD"].head(10)

eur_sample=df[df['PriceCurrency']=="EUR"].head(10)
for sample, ns in [(usd_sample, "storefront-us"), (eur_sample, "storefront-eu")]:
    vectors = embed_batch(sample["text"].tolist())
    index.upsert(
        vectors=[
            {"id": row["Id"], "values": vec, "metadata": to_metadata(row)}
            for (_, row), vec in zip(sample.iterrows(), vectors)
        ],
         namespace=ns,
    )
    print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1024, total_vector_count=761, metric='cosine', namespaces=2)
DescribeIndexStatsResponse(dimension=1024, total_vector_count=771, metric='cosine', namespaces=3)


In [18]:
# Querying is scoped to one namespace at a time — results from other namespaces never appear
result = index.query(
    vector=embed_batch(["a comfortable running shoe"])[0],
    top_k=3,
    include_metadata=True,
    namespace="storefront-us",   # <-- only searches the 10 records upserted above under this namespace
)

for match in result["matches"]:
    m = match["metadata"]
    print(f'{match["score"]:.3f}  {m["text"][:70]:70s}  {m["price"]:>8.2f} {m["currency"]}')


0.462  Saucony Men's Kinvara 13 Running Shoe:When it comes to lightweight spe    600.93 USD
0.315  Dingo Mens Dean Harness Pull On Casual Boots Mid Calf - Black:A harnes    519.66 USD
0.278  Sony WH-1000XM5:Sony WH-1000XM5 headphones provide industry-leading no    349.99 USD


In [19]:
# --- create two small demo records, clearly marked so they never get confused with real data ---
demo_vectors = embed_batch([
    "Demo Product A: a placeholder item used only to demonstrate update and delete.",
    "Demo Product B: a placeholder item used only to demonstrate update and delete.",
])

index.upsert(vectors=[
    {"id": "demo-1", "values": demo_vectors[0], "metadata": {"text": "Demo Product A", "price": 10.0, "demo": True}},
    {"id": "demo-2", "values": demo_vectors[1], "metadata": {"text": "Demo Product B", "price": 20.0, "demo": True}},
])

print(index.fetch(ids=["demo-1"])["vectors"]["demo-1"]["metadata"])

{'demo': True, 'price': 10, 'text': 'Demo Product A'}


In [20]:
# --- UPDATE: upserting the same id again overwrites it in place, no duplicate is created ---
updated_vector = embed_batch(["Demo Product A: now on clearance, price reduced."])[0]

index.upsert(vectors=[
    {"id": "demo-1", "values": updated_vector, "metadata": {"text": "Demo Product A (clearance)", "price": 4.99, "demo": True}},
])

print(index.fetch(ids=["demo-1"])["vectors"]["demo-1"]["metadata"])   # price is now 4.99 — same id, updated content


{'demo': True, 'price': 10, 'text': 'Demo Product A'}


In [ ]:
# --- DELETE by id: remove one specific record ---
index.delete(ids=["demo-2"])

try:
    index.fetch(ids=["demo-2"])["vectors"]["demo-2"]
    print("still present (unexpected)")
except KeyError:
    print("demo-2 is gone")
